# Computational Pathology Framework Demo

This notebook demonstrates the key capabilities of the multimodal fusion framework.

## Contents
1. Basic Multimodal Fusion
2. Missing Modality Handling
3. Temporal Reasoning
4. Stain Normalization
5. Model Interpretability

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models import MultimodalFusionModel, ClassificationHead
from src.models import CrossSlideTemporalReasoner
from src.models import StainNormalizationTransformer

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Basic Multimodal Fusion

Demonstrate how the framework fuses WSI, genomic, and clinical text data.

In [ ]:
# Initialize model
model = MultimodalFusionModel(embed_dim=256).to(device)
classifier = ClassificationHead(input_dim=256, num_classes=4).to(device)

# Create sample batch
batch = {
    'wsi_features': torch.randn(4, 100, 1024).to(device),
    'genomic': torch.randn(4, 2000).to(device),
    'clinical_text': torch.randint(0, 30000, (4, 128)).to(device)
}

# Forward pass
model.eval()
classifier.eval()

with torch.no_grad():
    # Get fused embedding and individual modality embeddings
    fused_emb, modality_embs = model(batch, return_modality_embeddings=True)
    logits = classifier(fused_emb)
    probs = torch.softmax(logits, dim=-1)

print(f"Fused embedding shape: {fused_emb.shape}")
print(f"Predictions shape: {probs.shape}")
print(f"\nSample predictions:")
for i in range(4):
    pred_class = torch.argmax(probs[i]).item()
    confidence = probs[i, pred_class].item()
    print(f"  Sample {i}: Class {pred_class} (confidence: {confidence:.3f})")

## 2. Missing Modality Handling

Show how the model gracefully handles missing modalities.

In [ ]:
# Test different modality combinations
test_cases = [
    ('All modalities', {'wsi_features': True, 'genomic': True, 'clinical_text': True}),
    ('WSI only', {'wsi_features': True, 'genomic': False, 'clinical_text': False}),
    ('Genomic only', {'wsi_features': False, 'genomic': True, 'clinical_text': False}),
    ('WSI + Genomic', {'wsi_features': True, 'genomic': True, 'clinical_text': False}),
]

results = []

for name, modalities in test_cases:
    batch_test = {
        'wsi_features': torch.randn(1, 100, 1024).to(device) if modalities['wsi_features'] else None,
        'genomic': torch.randn(1, 2000).to(device) if modalities['genomic'] else None,
        'clinical_text': torch.randint(0, 30000, (1, 128)).to(device) if modalities['clinical_text'] else None
    }
    
    with torch.no_grad():
        fused = model(batch_test)
        logits = classifier(fused)
        probs = torch.softmax(logits, dim=-1)
    
    results.append({
        'name': name,
        'probs': probs[0].cpu().numpy()
    })

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(4)
width = 0.2

for i, result in enumerate(results):
    ax.bar(x + i*width, result['probs'], width, label=result['name'])

ax.set_xlabel('Class')
ax.set_ylabel('Probability')
ax.set_title('Predictions with Different Modality Combinations')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(['Class 0', 'Class 1', 'Class 2', 'Class 3'])
ax.legend()
plt.tight_layout()
plt.show()

## 3. Temporal Reasoning

Demonstrate cross-slide temporal analysis for disease progression.

In [ ]:
# Initialize temporal model
temporal_model = CrossSlideTemporalReasoner(
    embed_dim=256,
    num_heads=8,
    num_layers=2
).to(device)

# Create sequence of slides (simulating disease progression)
num_slides = 5
slide_embeddings = []

for t in range(num_slides):
    batch_t = {
        'wsi_features': torch.randn(1, 100, 1024).to(device),
        'genomic': torch.randn(1, 2000).to(device),
        'clinical_text': torch.randint(0, 30000, (1, 128)).to(device)
    }
    with torch.no_grad():
        emb = model(batch_t)
    slide_embeddings.append(emb)

# Stack into sequence
slide_sequence = torch.stack(slide_embeddings, dim=1)  # [1, num_slides, embed_dim]
timestamps = torch.linspace(0, 365, num_slides).unsqueeze(0).to(device)  # Days

# Apply temporal reasoning
temporal_model.eval()
with torch.no_grad():
    temporal_features = temporal_model(slide_sequence, timestamps)
    temporal_pred = classifier(temporal_features)
    temporal_probs = torch.softmax(temporal_pred, dim=-1)

print(f"Temporal features shape: {temporal_features.shape}")
print(f"Temporal prediction: Class {torch.argmax(temporal_probs).item()}")
print(f"Confidence: {temporal_probs.max().item():.3f}")

## 4. Stain Normalization

Demonstrate transformer-based stain normalization.

In [ ]:
# Initialize stain normalization model
stain_model = StainNormalizationTransformer(
    patch_size=16,
    embed_dim=256
).to(device)

# Create sample image (must be divisible by patch_size)
input_image = torch.randn(1, 3, 256, 256).to(device)
reference_style = torch.randn(1, 3, 256, 256).to(device)

# Normalize
stain_model.eval()
with torch.no_grad():
    normalized = stain_model(input_image, reference_style)

print(f"Input shape: {input_image.shape}")
print(f"Normalized shape: {normalized.shape}")
print(f"Output range: [{normalized.min():.3f}, {normalized.max():.3f}]")

# Visualize (convert to displayable format)
def tensor_to_image(tensor):
    img = tensor[0].cpu().numpy().transpose(1, 2, 0)
    img = (img - img.min()) / (img.max() - img.min())  # Normalize to [0, 1]
    return img

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(tensor_to_image(input_image))
axes[0].set_title('Input Image')
axes[0].axis('off')

axes[1].imshow(tensor_to_image(reference_style))
axes[1].set_title('Reference Style')
axes[1].axis('off')

axes[2].imshow(tensor_to_image(normalized))
axes[2].set_title('Normalized Output')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 5. Model Interpretability

Analyze attention weights to understand cross-modal interactions.

In [ ]:
# Get modality embeddings
batch_interp = {
    'wsi_features': torch.randn(1, 100, 1024).to(device),
    'genomic': torch.randn(1, 2000).to(device),
    'clinical_text': torch.randint(0, 30000, (1, 128)).to(device)
}

with torch.no_grad():
    fused, modality_embs = model(batch_interp, return_modality_embeddings=True)

# Compute similarity between modalities
modalities = ['wsi', 'genomic', 'clinical']
similarity_matrix = np.zeros((3, 3))

for i, mod1 in enumerate(modalities):
    for j, mod2 in enumerate(modalities):
        if modality_embs[mod1] is not None and modality_embs[mod2] is not None:
            emb1 = modality_embs[mod1][0]
            emb2 = modality_embs[mod2][0]
            similarity = torch.cosine_similarity(emb1, emb2, dim=0).item()
            similarity_matrix[i, j] = similarity

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(similarity_matrix, 
            annot=True, 
            fmt='.3f',
            xticklabels=modalities,
            yticklabels=modalities,
            cmap='coolwarm',
            center=0,
            vmin=-1, vmax=1)
plt.title('Cross-Modal Embedding Similarity')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Diagonal shows self-similarity (should be 1.0)")
print("- Off-diagonal shows cross-modal relationships")
print("- Higher values indicate stronger learned associations")

## Summary

This notebook demonstrated:
1. ✅ Multimodal fusion with WSI, genomic, and clinical text
2. ✅ Graceful handling of missing modalities
3. ✅ Temporal reasoning across slide sequences
4. ✅ Stain normalization for color consistency
5. ✅ Interpretability through attention analysis

**Next Steps:**
- Train on real pathology datasets (e.g., PatchCamelyon)
- Evaluate on downstream tasks
- Compare with baseline methods
- Conduct ablation studies